# V3 LiDAR auto detect trên Google Colab GPU

Chọn **Runtime → Change runtime type → GPU (T4)** rồi chạy các cell theo thứ tự.
Notebook tải checkpoint **PointPillars nuScenes FPN** và **CenterPoint nuScenes Pillar02** với đủ 10 nhãn DOCX:
`car`, `truck`, `bus`, `trailer`, `construction_vehicle`, `pedestrian`, `motorcycle`, `bicycle`, `traffic_cone`, `barrier`.

Dữ liệu đầu vào là một frame XYZI float32, đơn vị mét, Z hướng lên. Model được huấn luyện bằng nhiều sweep của nuScenes;
kết quả một frame cần kiểm tra bằng point cloud và camera trong V3.
Notebook chỉ chuẩn bị model và tạo worker; point cloud của bạn chỉ được gửi khi bạn bấm **Chạy model** trong V3.
URL HTTPS và token ở cell cuối dùng để kết nối từ Mac. Giữ notebook mở trong khi annotation.
Mỗi lần chạy tạo thư mục riêng, giữ lại dữ liệu và thư mục cũ.

In [ ]:
from pathlib import Path
from uuid import uuid4
from io import BytesIO
from zipfile import ZipFile
import base64, subprocess, os, sys
subprocess.run(['nvidia-smi'], check=True)
ROOT = Path('/content') / ('v3-lidar-' + uuid4().hex)
ROOT.mkdir()
PAYLOAD = 'UEsDBBQAAAAIAKtgQV0itOkLzQUAAOQMAAAYAAAAdjMvY29yZS9saWRhcl9wcmVzZXRzLnB5tVfNbttGEL7zKaY8tFIhUtRPAluJA7i2UBhxrCByjDSCQKzJpbkwuSS4S1ly4FOPPeXQQ9FLjaAomksLtEAB+1YFfQ+9SWe5lEyriVu0qAGRy5nZ2W/mm51dm6bJ86FHORXgU0k9mWTgJVxmxJMCYiK9kPETkCEFIXOfcvmJgN3BzguQZJrwJJ7ZhvGY0hQI54kkkiUcfCbSiMwgyXyagaApyYikEGRJDF5IvdM0YVyCFxEhLMZ9OtWmtmGaplGYuW6QyzyjrgssTpNMVvwLwyhlPI/TGRABPDWMz5/v7fb39w767v72Z/39IWzByAD8Mz2SmQ0wZZZ7p2pwnAv9TVhECxWGLJRauXcnNGReRJU8pT5FBSPcbGhfcYIp8mal/pithugtCJjnoiutIlnGlHdjbBw8H+70D/pDd2d/ezjs3wHtQ0D+CvnG/924qiGso0RoT57s9g87u+5R/9lwb3CAyMxJy+7ajmk8fTY4HOwM9gtZx4qYTzKrZRo7g8Gz3b2D7cMiEFPLz608tU5oElNcy7OwpCTNLH9q+TPLP7dm5MzKiI8ohHLcH/YP1eRXGnxRDymLIpIJi+eiKEizV6oLE05iihLzqTJ9qk3h999gVb44bjnAw/mbJVc6wQkP2ImaqUeiWV3s1ocbTtwg5a445haJIndjety12lMXAVkd305nt/yu6lj5DqVMRa/Z9JMzHiXEt5OU8jiOyLHtJXEzjvXmQlI7fhPz69iOGyc+jdbgIIJbiKpwutMNd4Xmn1u6bafdcjba992W093s3LcCj7Q3N72WncrwdkS4xRARxtOtiDPCT1TmR9Y9pwHlowHqrX6dsba9KOtQE19A+xsmdwrLgs9/R2RlqerY1Slx2q6gaOqrl8pOSIlveSzDfcFjodlVuwSrte3Q/5nlD2F12hrtGtYbmIpLjdItUf7X+aog2s5Gq+U6ndZGt2u1Nluks9Fu31EQ995fEC27raqhfGFBFCP9rBTGhWEYPg2gyIUuW1HTrx62bptjA8nIrFEaML+Hh03WADSgXDA5c4VHItqDALMusW1gautgParM7enqM81tn6QSjjrw4ouXeyAToMQLK6fOA32W4aEWUUucqZNLsphCEgSCSmACzmmW2OogUh5FSj1csOxYoyXAcaHUIaAacRBR4CjDaoAvZyndQkWBudOuFzNYUE5C3CyGj7agjUffUiZCktJRa6zkXSXHEw+QxtJpfSlCr0wEjDNJlyob932t3luxhOeEoHBEopz2syzJarpx4pmb5D6k4eLqDQNv/gv+Lj1495otrr+MocQK0+ased5cpR/CxfVPOT6vLrltriJZh7JGVwF3TQYPt8C5C+Xeak1tXyKNFtffcgj/uOTgwGT+3fsQZVTkkdRsKA5FrVbJXaOgcnRT1OP6h0jSjka9BvS6Y/SnPaypOmP4dGs9vMKGTlEmb1eFXlvvnHGZQJznUzSr1SoLottHW6WLEX7V4WNY0z9cqju9cV0TT6ZMbLVuEaPd24TP7i6Lx+H8V7zgqVJYVoHMEpSozF/GMGF6Vz6A06WWoKViYrK4/gri+Y+yYERmi+vvPXiJLP2s2DqBaP62Qg7e5vgyfxrcuGwLE6KuEJK62Ehxd8UkreEvxS3aw6ukJ0dFM8DHuAGFCXaNiIlCPi76wAHeZlYdYMCjGd5c1Y0UM4X32NVVVd84VVNQPWB1ud052j5U1RFF6guLVpyydNUBML/JGfXVZUWdXLZHBA2SyMe0ghJAoHalGjBewrsoJiq5SPLMow1EkJ2o7sKhjMzGDROLKjXIm7Yu6YP1++yN6XuZDMwDdWRqtuBU0yrDfHH9jVfc2PEQ1gtclJyUq5bYsOeYrqtCd10Tc+dDrdSUeHRsKj9aXsmEmlyCR3WZMRttalpaGOhZ9b8No/jvYjZ/m4O3uPohh/klD2GKxYhlN7/CoIqrwSqYB8tY371eXH/NQOAmg1d6sQtVfn8CUEsDBBQAAAAIAKtgQV2IzDccQwoAAJgbAAAYAAAAdjMvY29yZS9saWRhcl9ydW50aW1lLnB5rRhLb9vI+e5fMWUOJQOFtpNsD85qgaytTY06ThAl6UMwiBE5EgcmZ1gOaVvr+tCfsOeiaIKgWKBdoAt0iwLOoQct9n/on/T7ZoZPybBT1AeL8/reb8dx9mkYs4g8f37AChYWXIpHB4SLGcuZCBmZyZwUMSPFuSQzfgE3RTkOmWCKwLvwNJNcFMp3HGdrlsuUBMGsLMqcBQHhaSbzglAhZEERsNrasnsxVXHCp9XS/MCGXxY8qXalMiAzWuDlCt5LWFZXijhnNOJibm6WeYJAcvb7kqkKLO7KjAl7peRRfQDfj2uaRJlmC0IVEdmWuesnPKJ5kOVMsULV+F+NxqPX4wE5fjPeHx2PxsH+0dPxeAQ7z94cHoyODo9HwdHTL0dHsJPKiCWBEdLW1laYUKXIEYJ9VYqCp2xvi8BfxGYgOS54EQSuYslsQHKWyT3NLPkDOZaCkaH+GZBzxudxoTYdegYc/uVSFrCLd1xQCk9AJR6IRsnkjLmen1FQcKEmuyf1E0TsI154p39A9/q9VP4crEOcuc7bRwGYyuj1o4Pg1YsXr52BQbRNnHlWBucyP2X5dsRA3hHYD2dqO02j2rIix/O66CwzgLH62oz06PDg6avg16PDZ798Pd6MtWWPa2i0HhDL5VX3IJHhKWzXhuQfwYbrbdVqQetTVidGmzzaI6rIPfLgC1KUWcImSO9AU33SKEBlLEQFGHOZVG8bcecMHEW0pL6t30ycUIoZnzsng66I6uOaT+fEz1WW8MJ1tkEmu97kAaizJl2B25WGdk1swlUxiXhYtKgEtpUUQCc4cLVX5IvmAv7xWc9F/RkXUYDkuI7WL2qWcKWNsPu2i2Q//ul7SsLlO96LOLv+Y38HMEsxJ+nyBw6fP32/+vhnWD57+cZ3OjBBlRuwVDFB5mF806Eldv14Vh35QXDGcgVUQQj7GZCsKXPW0XUZ+5KGp2DzRCzfLUi4uv5WbGKwx4fhBXBDhDR0+2EZUZ+rgJ5RntBpAq56G+p9jQ2kRI7fHh4cPt3ef3Pw9Al5/nIMQlz+DQihITmNlz+ALMN4df1+QaYtajfStEm+HTGGZ77MFCH3yEgoiPc6SYQyzSDQRAQpIOwCvBcFaV8pvwbJLkKWFcQ91AejPJf5gLwY2w8bGvXKw4jM8KtLUM3+zHm+Zi7AJpqZWl3/C/xr+U7M98hlsciYqyF5oGNBUwiIV07LEVSZYMScNB6Kye+ULQbGl7monNnnBUtVXzPGbQetvAjQtAebGAKQvM6DlCsFIQduof7Nc9Q9BmvXw0Co92twzVkHDJgt2IqO2Vom8M61jlasrj9wC7kVH32yb+wAUhvmgcCERx/yn7GYdAmH2u+0Z1g60SyIM26kam+jtn3H6+kHxenTDDOBe+nwyNkzsnRQ9LAwwUwvINI5OjUyBQcYptx+cvXgCoboBVxAqVheKRoxLC2J5hIcwC0jFthJJI2YRY9KbCWEK68fjQ3drfBvJNRLAMpQOYE0cNKEVlw1JgEl0YE8F4idyNmMh5wm7ZrpCaKkQA8swCzLrADHiewLpW0vZ6GEYLTQ5VUFt50T/PQ04rlrc/nwdV5CfcAugJhAnuplw6Asi6zUmbZn4RVTKJuGwU8y7eqZ108bmwx7Q36gHCyr7fbuzPmViVjF8rsUgsvq+q8LC4lcmt8rMGNMJDp3doPtWZVOMCj8KewbJ7sA2wNxB1iHAictz1Dg2XVSfdAk1R5fm5xyna+Iz7EOHVb1rq9i+vCzX7it12jTwXRRMAgnnh+zC/Om5+ItaZpzoJNCRD3nUCh1uLkpW2yS8H6jzXD5T8OTKlOiKId42RAJgv4KOCQ/foPi/BCSOV99/AcR83IB7r8m3I3JA+idm1jXq2lmTguTj+Hgyq+84MGlLtFdLZir9Tx1k3zXhYeCqhoBd60k69ZUA4JCAncZ7j7c0fkHgkIG7Qv4lmXD13Cci6mjz41rbZb8eYyiC+NSnJK9YQ1KK97d3Xn4mNwn+HOD4vDPgPfPc0g7roa0zmBXJH6ZRfTmy11Tahnd/8+s0I5M9jlbffxjVX78+M3yWsyfWANCt8FL7zm5tIK96luTJbdlIjq8rSXf2ykyhg7U/J02lozYJZmuPv4FSgi+uv6PwJz2nYiR4vfhp1JamQeEYzDklp93r9pQXGVHyBvtq2s5yV5vkpJuz21KMskEkmLmC2grc7ro9SkD3dyoWCawnoFbFQOdcqA0KxaBCmnC7D440q6/c2OrgJVAlSrQfJp6qFeaaRW8pUlpFeA8x2cki7U9JMt35CUS/ZInCc0VieXq+t8h2WeYB/VJPWNoy9ga7Q75fNhwhIvd29DvY7qI9DijeWipMR3HZAcC/UkbG42oTsjDTgvvmp9GwmuibPWd0H1hXy6gDnaxYNTJVn9URYht0DzdYMHBBMukEzIckvWEatnXUCe2FDrZxHnH9pvrWBSdNODukTHLoSLhXzNdMdcTnydEVyyMhnE73UuUHlZbOSsBCS+aYl7H1rqZ7pKk5yhpygS4Bau6B62P+aZ7pgOjGa+nLXomoqUxaGgMzERB5v203LfPVqm3Hi7+t6JGv5xhIjNs+Ei6LgC0G+s9b0NkuG0a0CjmTWYqVhAKBBHwD3rGyG9++7tDW+0Sdc5YBo4P5fI2qhqUnGat7qoBdSBtsQyJMWPk+OIx2quEj8+wTZjRac5DNFOB+XgqyxxztIa/AV7GM+hW9bBpsjH4YsBwsc8aOkfAgvZl9RXIB8sHqKVCKfMosBdwoAN7yGsQ8XRYDTeoEKAuTMNgaBtPvMFt2A3mV1TMGaCGsOLYQBmEiSyjIMcTC1d/3wkoNM2PDqDMPBQYjgEkNBRqOHHszGkTiHX9gu34UO0VASRoqhWd+/ipWOG3BFx9rhtSY9KNFcGDxlFcQIGVSiepDCBxnPEQ2MAJw95OL3nph5X596F3WwHTp8Fd3afpSxUDQQodDU7t3LqdWx+TemtNQgK1lL3v4chldweNE+B1dnHdH7DerZdo5/9qDgL1rilGEJuIlx8ECVcfv6VkDhWnlnw/vytwsgRKwEALux+JXBujbN7oPoUmEu2I64GFgePjXsAF+C6AUUFvJjWVF1rGzUt/qvf6F++ZzufnOFUGhyLznJ5hMgp1NiX0TEIzB+LkEIB1/Ih15U1oOuXAasFZz9HNQ8St8fldgD521WEMlWKYlfBfD8179baCnNK8BwdWd3m1oOfNI1jcCVMo876YzB6I6S4AEjo1Y+EWALN3RwBmXNBrqvEPc70R2UALZIAMDgzFA4MXE9TXPHOtxM09pS8qe1PZq2qDoduCAOo+rmbo/ZCB8I0eHOkv8nmr3qk3v+gXTLXmJfiMKNdDzgZMk/sVc/dr7k48nyaJHVxlPhULF8+wRNu5oVq/DaUu9sBtXS0FDxhqx4pPhGl11cykNFBnrwppkwaRHkhpBvHYWH4hdcDDQRSyhRMs+GlvgwwcW0q78I1bYV161idGSVd9O2pNn9TWfwFQSwMEFAAAAAgAq2BBXWg9xfmKBQAA4w0AABUAAAB2My9jb3JlL3BvaW50Y2xvdWQucHmNV21P20gQ/p5fMfXpJLtnfAlFpypqKkGhUnRtQYKerhdZ0RJvyqr22ue1Wwzqf795WRuHctBIJPbuztszz8wO27osYL3etk1b6/UaTFGVdQPK2rJRjSmtm0z8min7J9sWVQfKga0mk/eHf6+PPl2cnMMC9qdTeA6z6f6B/+Hds9Plh4vz9fHy/Ozd4Sc8NptO19PpdDKZZHoLlaqdXlelsc0mL9sszFSj5nDZNdrFsDW5tqrQc3BNHcHeazSa2EzVtermE8CP2QIeYakIXsPgj+zSp1bGafhL5a0+qeuyDoMzsgZsDvT1RuvMsfPvj4KIxVy73Zpr9LW3n+TlN12HUVK7KjdNGCRBDLNotTdLey96mQUEl8YGd/Z3PPwVZn/cbT3s3tHyAxStQw9L2yhjYZuXqnmxD9dxF9/E6Lu2zjQd1HpT1pnzTtOHcXToOMK0xeReolPoN9mOIWu6Si+CV9uDAAPR7kpVOtybxXAQJZuy6kLRo/PdaKpNNopmsLD2mdtIynpZpx9D/tTmHSQogxzLIEGgRKEkw4GqNZquiGc683GhN0hHxlCMR1DWvIRBGrc11jS630pUnofRzybfONBFhUiiQg820rq0e6IUvpJcj2+tsUq8vy5RjtAMCWfJTgyE4eKtQggiT+57GHlaP8hj5LdWBeJqyuSITi1PR7heaZXpeg6Z2TQrPBpDbhw/pSnK3H7nU1sMA4vYYtT2sw6x0EZI5MZqPCp2MP0qo5UwGvOUccbVpxh69ubYuwRX1AhKOD68OMRq0Xk2ZqOqmSqkMsmQrJkOA+U2xmD1aFLlFgE6hEEhI+mhwhKTCtvxSxQRZchDfltNU5Sgp2+muQqDX4Jo12vxbzUcRlZRBRNcsjabpw/buC/CVUABBrsWLhHELwI8BU6RitHks8YW8XZ58u74HCNdpTtEvg2ucTHo6Osm+I4Udu2lQwnR8jh5Efha/9saLF9sB9DFcOOte9w3ZYv0RhrZJhx7I22YvAmmQRpheINTIvJqAVOpA3p7jQ2RujT9PebQ0mKNmAzIMTEhCrw3Bab8HiwMJLkhXvSdtXdGJBBw4cnY9jfpa1huWXPdhMLkvq2NC9FmhbGL/R0KoXjCHQ+eLSBkH2PuKR716Gc4TwUJWakdJ7JQzebKxzZive8Ta6dzvWnWoj9E+7HP1G6sz+TCUHUXPJV5LjK+GS41MD6UsNZuyqJCQjidgdfkrzFzo4mVKyLD14j7w1fqD+N8nC//ORGSSjkQmvepfPHp7GREZEbvScVvTj9+uOBMz4IUJ4Ix2MOtSYvsZkRAjI5QZPTK7vzfpjhyf/cpGAUjSQYUulGcVrwLDCOZ60YPxZQxFrdYzQHdk3Pg6zMGWXgpCy95YckjAS6Y2fC+zwfM/rAgKoyo+NhLtLPhXSTa/WFBJNqDQHp8U3d38TH1pSr4MVyFNK3E4vgq/GJsFjMNohT12TjiJoTVPeOrGsJIkidSdDAGEbKUzhtsyAKqbOIP5yP2FIg8IWiIqhr4U3cMMw2H3N4fycQ2+GiHm57bh+SD9M/hlsW/B+gdjaj8NpErpaP6373I7roHDwlyhFkhzey5AJXglV5QGD9JkCer3c9fP4xb3oO+NfG3h2zB3zsDBcpuyrwt7Brvs82XcOXVrigp6ZAeyke4c3Xg1zALBpJZf05ylsJvEK5S2hgdvNsXCqzQ/o2uSyfVhCkdppcfOth8NLb03Wx+N4o8ONhIbNwtUMNq3sslSDN9zXyNHovS9wqvJlF4J9ssfEjVGI2noh6CJsagsih6NCfeeo9MZnBIUZ382+L87DkGh5FgXPiLb6Y0HSY9w7dioa5DnL6ph5I9bfJwPOT+Dj/++3TPTTm6ms9FZ5rUmMIsfBElTUmGsTD+A1BLAwQUAAAACACrYEFdtWp9tU8EAABJCgAAFwAAAHYzL2dwdV93b3JrZXIvc2VydmVyLnB5pVZtb9s2EP7uX3HgJxlTFLvpgMGbiyWt1xlNY8N2+rIgEBjpHHORSI2k4npZ/vuOlGRbzrpmmL5IvOO9PXzuKMbYaWlXKK1IuMUU3k4vYa30HeoIZqUcQHkvEqUl3BZlXCsM6nvUA14UcHRUKG3hh17/JGKMdZZa5RDHy9KWGuMYRO71XEpluRVKmk6nlq1ynjTfisTedMmN5YVo7H6h5el0HMKvi8V09CXBwvkIYYZ/lGhsZUMWOkNrMUqUTEqtUSabxoMuZSxkbFcaeVooldWBqCaMMpFyHRcaDVrTWExnk8Xk9eQ8hNeTyezN+OJ0MZqH8PZy/GZ0Pr4YxeenZ6Nzkkxno/loMX/ij0JakWPj79wJZ5Vsb2+hhLRJpsq02fj+9FN89tkHK7g2GO+2dDqdFJeQUBEWYwI+qIMMWu7hL7hQEmHoXyFYdYd0hMbqtqYLR68abAcdoKfJebj9UrrlOuj6fd4j7areYtl8GKADrkJgZsjaRLdoUd4H7MNJ/HEyezeaxYvJu9EFC4GxyllFJ/JWpxJYYTMckgVMXelTkWUEBBzDayIoai/0DP3oLcmN9+Og4cRipcWfGOiKG4OGJN2qQvdQvi7NCpat1NfPBWXdYlnwfe+Ecp2jhYMS4AaXdITg+kDIW6gjmqiuyz2mLIpMUEMNG3W0Igai9sAErMnXN8UeJntpugYhSudEBoxTcUs+gsZtRBxXKZ1KCEt2hrRDw4Ov65Ftdd1v1/iy16fYY3nPibt7zV9h1AD8c934PvNjqiOzqzpfh30l+Dfgn5zOrliNNCokPLBCK6sSlbHBXguyRCmdCulobzbGYk7qVl+yJOPGoCH50xZlOQGROV1N64iGhS2Nw40JWZSWVMzQIWZ4ZNaIBXz6/NsYlpni9uQF5GhpOPwICc9RcwOEsz8aQ3xkj21wCmUcOkIuUR8/+MCxSB9rnLjZyMSj5TcEjd53ZwiH2FHv0sQyK5XRDp8NEakXESGpBVAaYTexSXiGO20/6j0PcaJXE90XI2Qzyp7Dl5eE3KW8k2otQZbzBCUaqstiYpV+SuIe/DTc1eIW/edE6VGUxdYqL42lpoO1sCvK9oq0/eu9WMmqlHcmpMFKp0xQXF2H0NtB4aGnjq32uXqbliTskefBQaNUbr4bQoYy8Dbdlp5Kq7a82s3stoevFtZ3I6WaY9XsR9JhauBFrwfvz1g7UFVXRPMeZXqYidWbdlB/XRgq//D2CG4Yi36nZeXCOPIvNVE6uhHyIOSN+oLOB19z8Q+3Z3PvRJ7FYR0z3BJqj7dPqNoO9I2+bxySdOf7/04DXxyp/JvWa64l9b4TXbH5/hTws2EABWriTc5lgpDzDaRiSWWDv8W35M/LzIra7oZmL81tTafGrh+3BaNnAHzgWYkjrYmKNEzQfTyzGYiogd/f7VbR/eLQf31X/9cI/qL7SoTO3llVg47+Rdyf33D/b6Tb+RtQSwMEFAAAAAgAq2BBXcbE4D/yAAAAowEAAB8AAAB2My9ncHVfd29ya2VyL3ByZXBhcmVfbW9kZWxzLnB5ZVC7bsMwDNz1FQQnG421dAvgoUO2DoHTnRBs2hZqSQYlp+jfV36kaFAtEu6OvDshYrN46CU4uL+eYf5OY/BQORjmhb6CfLLoWXg2wuRCx1PUiKism4MkMDJkJrLaFrRBWE+2M0J5JHKKcOiuzeV2+bj9k8nik3X8kL2vYLNjStkeiLxxTAR1DUjkjPVEeFaQz2YsUP+G0G8yLI59um5MUf6RadN1ZA6+wKrau+AJfEZjjS/52Y7BthzrycZUHJHLE3Tcm2VKz/C+fJ3NCQ6P7Vpd4uHdB8lcGsH6p25F+fjTYlXrPUy599pCi80x19FS/QBQSwMEFAAAAAgAq2BBXSAuK0XaAQAAPAMAABcAAAB2My9ncHVfd29ya2VyL3ZlcmlmeS5weYVS22rbQBB911cM+7QCWbhxW4KLH9JGhFDThjh+CmFZ747qxXsRewkRIf/elSUnvUEXBGLmnJkzM4cQcnWzhWDcASFiiOBa2Lm4B49cg9ijOHRO2Rg+QdyrANJhAOsiGOQheQQuRPJc9DUhpFCmcz6CTabrgWdgV7TeGRDOY62V5J51HgPGABP05rbZNHebCq6215fN+vpbw9YXn5v15i+iTzYqgyfiegjejrGiKCS28IhetT2dgMvfIOWygPy8/QGrLKv23Epn6kzjSUeW4/T9WTliHJcjSDidjGUhcnGg9xlTJ6ta5w2dnX2oYPjO5/N5WcH/cjYnuKazd/XHCur54p/E+UA78R7KmofYd0izkFY7HheTvowF4yRqpiQoe1rhOODwdu4pH2kF0yJqZVv0dBireiVmFYvylaFa4LanmXhPNN+hJg/HI+fqf97l2D4Dh9yxUfnW+Lg9rgLCtPXGe+cp2Vp86lBElCAx5p9cQmgeAnmT0PnsMtqSL9vLCzgqRisQvn9dwvNJ9AvQZ42Wjn1fIPQ27jEqMRPaJZlroFQiKmezXU/GBDUaFh+5TjxrKEkFrU5hv7rzCctsnjw+Y5YbZAxWKyCMGa4sY2QcbfLVr3aiZVn8BFBLAwQUAAAACACrYEFdvvQD74UDAAC1BwAAHAAAAHYzL2dwdV93b3JrZXIvc2V0dXBfY29sYWIucHmNVW1v2zYQ/q5fQWgfKg0RFcdFUSTQgCwt1gBFZrTJgGEYDJo8W4QlUuWLEv/7HknZcRMXaD44JPXc3XMPj3d5ni8MDMwAYYpIqzvmQJDFzrVakTmdndc3Dx+uyaM2WzBEKisFQskNAlfkr8UDMV452QPNsi9ekbXRPXEtkA0oMNGX0g5WWm8p+fgkrZNqQ9a6E2AshhRkYHzLNoAb5DAYsGBGEDTL8zyL3gbm2k6uiOwHbRxZ4Dab1truV9avBqM52OeTnc2yTMCaWHB+KIzW7jJal6T6Iy4uM4J/v5F75LtnSbgBpI10MBewLfFKfvNAgjnyNgQYb0PSNBqjibPOsIE0CVKT/HCWR8gzNYpmxX9IjMITcO/YqoMzkld9jr+DHMI/VNixrgvLqnLMbMDhGt0VB7clfvNj05zT9/TiIv//jPAW+La5Nx7KGNKPSOeZWuAkVe3HREiuQ7YIotIu17KDokxKnDb148ES1CiNVj0ohyghuSs0JpNOz8jDP8vFv/ef/r5b3t59vb/+/Hn54fZLE7jvpRliXdm8jOCb65tPH19hOAoME2Jy9+ft3SsYZpSX5Z6WkOboBvDgJ9qjAz8GAceAScIm8zIqnggG9UPtp1uwAOJHlc9CyOZIjcQjGSOPidCk++QzQkI94k0Xv+PN2iPZTxFNduWLComWv8QGwxyXk/L9sGuaGb14R9/mpzFOG942zQWd0fPDfpRWahXqbfaOTppIJeCp8iZatc4N9rKuhX5UnWaCIvNgSLXZ1I9tV3M/m73/Sci+5+NRxKrSqttVqBszuyZ8TKdKVwIGGzfrk0H1AKrvsS1Rrvs6WNYC+00KXkdCGKWOzGnr+m4iNM6PKmec5ydZVmYqFkQjbDP4ZWqJtYFvXhoIulvqnty+KLGranT8Co9ZAFJQXIJFlgIccIf6zkV+sKOhIStH+y2WUZE2dn/XoYku9fFzf1k8+UaGppHzTitI8mFU14blLO1XhikeD8YZfZuk3yuK1q1PKgZNqyjqj1RpihDfI/ItT/SgFwoeoPvbXHnZiSpNHPSZn07lxTvgsUWmBh9v9Go/FsJ9YwnYK8IsjpDpM+VesNDm2MhkF/ptgX7efIUOM8EWfzTAcLalqfbmCseQVK448oBdeClglByWivVQnJflqb77K6/4uRTokCbvstcCOhsdPopmnL92bHCEGTU1mOw7UEsDBBQAAAAIAKtgQV1nHrjvXgAAAHAAAAAeAAAAdjMvZ3B1X3dvcmtlci9yZXF1aXJlbWVudHMudHh0FclBEoMgDEDRPWdxMoSUdmO8i6PRsoBSCY7evs3yv196rjczQnjCw+UsZU9FmD2gh/iHVZSZgMC7bW461zTZxDiM6PqZls9RTMgbvFXrZRlelod8uzRtEwcgHEZyP1BLAQIUAxQAAAAIAKtgQV0itOkLzQUAAOQMAAAYAAAAAAAAAAAAAACAAQAAAAB2My9jb3JlL2xpZGFyX3ByZXNldHMucHlQSwECFAMUAAAACACrYEFdiMw3HEMKAACYGwAAGAAAAAAAAAAAAAAAgAEDBgAAdjMvY29yZS9saWRhcl9ydW50aW1lLnB5UEsBAhQDFAAAAAgAq2BBXWg9xfmKBQAA4w0AABUAAAAAAAAAAAAAAIABfBAAAHYzL2NvcmUvcG9pbnRjbG91ZC5weVBLAQIUAxQAAAAIAKtgQV21an21TwQAAEkKAAAXAAAAAAAAAAAAAACAATkWAAB2My9ncHVfd29ya2VyL3NlcnZlci5weVBLAQIUAxQAAAAIAKtgQV3GxOA/8gAAAKMBAAAfAAAAAAAAAAAAAACAAb0aAAB2My9ncHVfd29ya2VyL3ByZXBhcmVfbW9kZWxzLnB5UEsBAhQDFAAAAAgAq2BBXSAuK0XaAQAAPAMAABcAAAAAAAAAAAAAAIAB7BsAAHYzL2dwdV93b3JrZXIvdmVyaWZ5LnB5UEsBAhQDFAAAAAgAq2BBXb70A++FAwAAtQcAABwAAAAAAAAAAAAAAIAB+x0AAHYzL2dwdV93b3JrZXIvc2V0dXBfY29sYWIucHlQSwECFAMUAAAACACrYEFdZx64714AAABwAAAAHgAAAAAAAAAAAAAAgAG6IQAAdjMvZ3B1X3dvcmtlci9yZXF1aXJlbWVudHMudHh0UEsFBgAAAAAIAAgAPAIAAFQiAAAAAA=='
with ZipFile(BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(ROOT)
V3 = ROOT / 'v3'
sys.path.insert(0, str(V3))
print('Workspace mới:', ROOT)

**Cài môi trường và tải checkpoint.** Bước này tạo Python 3.10 riêng và cần vài phút.
Nếu wheel CUDA không tải được, xem lỗi ở cell này; chưa tiếp tục khi cài đặt thất bại.

In [ ]:
from gpu_worker.setup_colab import setup
PYTHON = setup(ROOT)
print('Môi trường và checkpoint đã chuẩn bị:', PYTHON)

**Chạy thật cả hai model trên GPU.** Point cloud tổng hợp chỉ kiểm tra khả năng thực thi.
Dòng `CUDA inference OK` phải xuất hiện cho cả hai model; đây không phải đánh giá độ chính xác trên dữ liệu bài tập.

In [ ]:
subprocess.run([str(PYTHON), '-m', 'gpu_worker.verify'], cwd=V3, check=True)

In [ ]:
import secrets, socket, time, json
from urllib.request import Request, urlopen
with socket.socket() as probe:
    probe.bind(('127.0.0.1', 0))
    PORT = probe.getsockname()[1]
TOKEN = secrets.token_urlsafe(32)
LOG = ROOT / 'worker.log'
worker_log = LOG.open('xb')
worker_process = subprocess.Popen([str(PYTHON), '-m', 'uvicorn', 'gpu_worker.server:app', '--host', '127.0.0.1', '--port', str(PORT)], cwd=V3, env=dict(os.environ, V3_WORKER_TOKEN=TOKEN), stdout=worker_log, stderr=subprocess.STDOUT)
for _ in range(60):
    if worker_process.poll() is not None:
        raise RuntimeError(LOG.read_text(errors='replace'))
    try:
        request = Request(f'http://127.0.0.1:{PORT}/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urlopen(request, timeout=3) as response:
            health = json.load(response)
        assert all(item['ready'] for item in health['models']), health
        print('Worker sẵn sàng:', [item['id'] for item in health['models']])
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Worker chưa sẵn sàng. Xem ' + str(LOG))

**Tạo kết nối HTTPS tới V3 trên Mac.** Cloudflare Quick Tunnel tạo URL tạm cho worker có token.
URL đổi khi tạo tunnel mới. Không chia sẻ token; chỉ dán vào V3 của bạn.

In [ ]:
import queue, re, threading
cloudflared = ROOT / 'cloudflared'
with urlopen('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', timeout=120) as response, cloudflared.open('xb') as output:
    while chunk := response.read(1024 * 1024):
        output.write(chunk)
cloudflared.chmod(0o755)
tunnel_process = subprocess.Popen([str(cloudflared), 'tunnel', '--no-autoupdate', '--protocol', 'http2', '--url', f'http://127.0.0.1:{PORT}'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
lines = queue.Queue()
def read_tunnel():
    for line in tunnel_process.stdout:
        lines.put(line)
threading.Thread(target=read_tunnel, daemon=True).start()
deadline = time.monotonic() + 90
URL = None
while time.monotonic() < deadline:
    try:
        line = lines.get(timeout=1)
    except queue.Empty:
        if tunnel_process.poll() is not None:
            raise RuntimeError('Tunnel dừng; kiểm tra mạng Colab và thử tạo tunnel mới.')
        continue
    found = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if found:
        URL = found.group(0)
        break
if not URL:
    raise RuntimeError('Không lấy được URL tunnel trong 90 giây.')
print('Dán vào V3 → Cấu hình GPU / Colab → Lưu và kiểm tra kết nối')
print('URL:', URL)
print('Token:', TOKEN)
print('Chọn PointPillars hoặc CenterPoint trong Auto detect.')

Trong V3, mở frame → kiểm tra ánh xạ 10 nhãn → **Chạy model** → kéo/chỉnh cuboid và hướng XYZ → chấp nhận box.
Khi Colab hết phiên, tạo worker mới và cập nhật URL/token. Quick Tunnel dành cho thử nghiệm; inference quá lâu có thể bị timeout.

Nguồn: [PointPillars](https://github.com/open-mmlab/mmdetection3d/blob/v1.4.0/configs/pointpillars/metafile.yml),
[CenterPoint](https://github.com/open-mmlab/mmdetection3d/blob/v1.4.0/configs/centerpoint/metafile.yml),
[MMDetection3D inference](https://github.com/open-mmlab/mmdetection3d/blob/v1.4.0/mmdet3d/apis/inference.py).